## 1. Configuración inicial

In [1]:
# Importación de librerías
from google.colab import drive
import pandas as pd
import numpy as np

In [2]:
# Montaje de Google Drive
drive.mount('/content/drive')

raw_csv_path = '/content/drive/MyDrive/TFM/data/raw_dataset.csv'
df_raw = pd.read_csv(raw_csv_path, encoding='Latin-1')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Información general del dataset
print("Shape del dataset:", df_raw.shape)
df_raw.info()
df_raw.describe().transpose()

Shape del dataset: (441456, 330)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441456 entries, 0 to 441455
Columns: 330 entries, _STATE to _AIDTST3
dtypes: float64(324), int64(4), object(2)
memory usage: 1.1+ GB


,count,mean,std,min,25%,50%,75%,max
_STATE,441456.0,2.996871e+01,1.603471e+01,1.0,19.0,29.0,44.0,72.0
FMONTH,441456.0,6.359676e+00,3.487131e+00,1.0,3.0,6.0,9.0,12.0
IDATE,441456.0,6.563745e+06,3.488675e+06,1012016.0,3232015.0,6242015.0,10022015.0,12312015.0
IMONTH,441456.0,6.416803e+00,3.492082e+00,1.0,3.0,6.0,10.0,12.0
IDAY,441456.0,1.449273e+01,8.335468e+00,1.0,8.0,14.0,21.0,31.0
...,...,...,...,...,...,...,...,...
_RFSEAT2,441456.0,1.824624e+00,2.360812e+00,1.0,1.0,1.0,1.0,9.0
_RFSEAT3,441456.0,1.887028e+00,2.351387e+00,1.0,1.0,1.0,1.0,9.0
_FLSHOT6,157954.0,2.290705e+00,2.518086e+00,1.0,1.0,1.0,2.0,9.0
_PNEUMO2,157954.0,2.412259e+00,2.778032e+00,1.0,1.0,1.0,2.0,9.0


## 2. Análisis exploratorio inicial del target

In [4]:
target_counts = df_raw['DIABETE3'].value_counts()
target_pct = df_raw['DIABETE3'].value_counts(normalize=True) * 100

target_distribution = pd.DataFrame({
    'Conteo': target_counts,
    'Porcentaje': target_pct
}).transpose()

print("Distribución de clases:")
print(target_distribution)

Distribución de clases:
DIABETE3              3.0           1.0          4.0          2.0         7.0  \
Conteo      372104.000000  57256.000000  7690.000000  3608.000000  598.000000   
Porcentaje      84.291504     12.970015     1.741991     0.817308    0.135463   

DIABETE3          9.0  
Conteo      193.00000  
Porcentaje    0.04372  


## 3. Cribado de columnas

In [5]:
# Se modifica para conservar solo las clases 1 y 3
target_classes = [1, 3]
df_raw = df_raw[df_raw['DIABETE3'].isin(target_classes)].copy()
df_raw.reset_index(drop=True, inplace=True)

### 3.1. Primer cribado de columnas (330 variables)

In [6]:
# Se eliminan duplicados, si los hubiese
df_raw = df_raw.drop_duplicates()

In [7]:
# Se define lista de columnas irrelevantes para la detección de diabetes para eliminar
cols_to_drop_1 = [
    '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'QSTVER', 'QSTLANG', 'MSCODE',
    '_STSTR', '_STRWT', '_RAWRAKE', '_WT2RAKE', '_CLLCPWT', '_DUALUSE', '_DUALCOR', '_LLCPWT', 'CTELENUM', 'CELLFON3',
    'CTELNUM1', 'CELLFON2', 'CADULT', 'CCLGHOUS', 'CSTATE', 'LANDLINE', 'NUMPHON2', 'CPDEMO1', 'INTERNET', 'PVTRESD1',
    'COLGHOUS', 'STATERES', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'PVTRESD2', 'HHADULT', 'HLTHPLN1', 'PERSDOC2',
    'MEDCOST', 'CHECKUP1', 'RENTHOM1', 'NUMHHOL2', 'CHILDREN', 'INCOME2', 'CDHELP', 'CDSOCIAL', '_CHLDCNT', '_INCOMG',
    'DIABAGE2', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'CHCKIDNY', 'PDIABTST', 'PREDIAB1', 'INSULIN', 'FEETCHK2',
    'DOCTDIAB', 'CHKHEMO3', 'FEETCHK', 'EYEEXAM', 'DIABEYE', 'DIABEDU', 'MARITAL', 'EDUCA', 'VETERAN3', 'EMPLOY1',
    'SEATBELT', 'CDDISCUS', 'SCNTMNY1', 'SCNTMEL1', 'SCNTPAID', 'SCNTWRK1', 'SCNTLPAD', 'SCNTLWK1', 'SXORIENT',
    'TRNSGNDR', 'RCSGENDR', 'RCSRLTN2', 'EMTSUPRT', 'LSATISFY', 'ADPLEASR', 'ADDOWN', 'MISTMNT', '_EDUCAG', '_RFSEAT2',
    '_RFSEAT3', '_LMTWRK1', '_LMTSCL1', 'CAREGIV1', 'CRGVREL1', 'CRGVLNG1', 'CRGVHRS1', 'CRGVPRB1', 'CRGVPERS',
    'CRGVHOUS', 'CRGVMST2', 'CRGVEXPT', 'CDHOUSE', 'CDASSIST', 'WEIGHT2', 'HEIGHT3', 'HTIN4', 'HTM4', 'WTKG3',
    '_BMI5', '_RFBMI5', 'STOPSMK2', 'LASTSMK2', 'USENOW3', '_SMOKER3', '_RFSMOK3', 'ALCDAY5', 'DRNK3GE5', 'MAXDRNKS',
    'DRNKANY5', 'DROCDY3_', '_RFBING5', '_DRNKWEK', '_RFDRHV5', 'FVGREEN', 'ARTHEXER', 'FVORANG', 'VEGETAB1', 'GRENDAY_',
    'ORNGDAY_', 'VEGEDA1_', '_MISVEGN', '_VEGLT1', '_VEG23', '_VEGETEX', 'FTJUDA1_', 'FRUTDA1_', 'FRUITJU1', 'FRUIT1',
    '_MISFRTN', '_FRTLT1', '_FRT16', '_FRUITEX', 'FVBEANS', 'EXERANY2', 'EXRACT11', 'EXRACT21', 'EXEROFT2', 'EXERHMM2',
    'ADMOVE', 'EXACTOT1', 'EXACTOT2', '_TOTINDA', 'METVL11_', 'METVL21_', 'MAXVO2_', 'FC60_', 'ACTIN11_', 'ACTIN21_',
    'PADUR1_', 'PADUR2_', 'PAFREQ1_', 'PAFREQ2_', '_MINAC11', '_MINAC21', 'STRFREQ_', 'PAMIN11_', 'PAMIN21_', 'PA1MIN_',
    'PAVIG11_', 'PAVIG21_', 'PA1VIGM_', '_PA150R2', '_PA300R2', '_PA30021', '_PASTRNG', '_PAREC1', '_PASTAE1', '_LMTACT1',
    'LMTJOIN3', 'ARTHSOCL', 'JOINPAIN', 'PAINACT2', 'TOLDHI2', 'QLMENTL2', 'QLSTRES2', 'QLHLTH2', '_RFHLTH', 'VIDFCLT2',
    'VIREDIF3', 'VIPRFVS2', 'VINOCRE2', 'VIEYEXM2', 'VIINSUR2', 'VICTRCT4', 'VIGLUMA2', 'VIMACDG2', 'ASTHMAGE', 'ASATTACK',
    'ASERVIST', 'ASDRVIST', 'ASRCHKUP', 'ASACTLIM', 'ASYMPTOM', 'ASNOSLEP', 'ASTHMED3', 'ASINHALR', 'CASTHDX2', 'CASTHNO2',
    '_LTASTH1', '_CASTHM1', '_ASTHMS1', 'HAREHAB1', 'STREHAB1', 'CVDASPRN', 'ASPUNSAF', '_MICHD', 'RLIVPAIN', 'RDUCHART',
    'RDUCSTRK', 'ARTTODAY', 'ARTHWGT', 'ARTHEDU', '_DRDXAR1', '_RFCHOL',  '_CHISPNC', '_CRACE1', '_CPRACE', '_PRACE1',
    '_MRACE1', '_HISPANC', '_RACEG21', '_RACEGR3', '_RACE_G1', '_AGE65YR', '_AGE80', '_AGE_G', '_RFHYPE5', 'CHOLCHK',
    'HIVTST6', 'HIVTSTD3', 'WHRTST10', 'HADMAM', 'HOWLONG', 'HADPAP2', 'LASTPAP2', 'HPVTEST', 'HPLSTTST', 'HADHYST2',
    'PROFEXAM', 'LENGEXAM', 'BLDSTOOL', 'LSTBLDS3', 'HADSIGM3', 'HADSGCO1', 'LASTSIG3', 'PSATEST1', 'PSATIME', 'PCPSARS1',
    'PCPSADE1', 'PCDMDECN', '_AIDTST3', '_CHOLCHK', 'FLUSHOT6', 'FLSHTMY2', 'IMFVPLAC', 'PNEUVAC3', 'TETANUS', 'HPVADVC2',
    'HPVADSHT', 'SHINGLE2', '_FLSHOT6', '_PNEUMO2', 'DRADVISE', 'PREGNANT', 'PCPSAAD2', 'PCPSADI1', 'PCPSARE1', '_HCVU651',
    'STRENGTH', 'PAMISS1_', 'SMOKDAY2', 'EXERHMM1', 'WTCHSALT', 'LONGWTCH', 'BEANDAY_', '_FRTRESP', '_VEGRESP', '_PAINDX1',
    'CIMEMLOS', 'ASTHMA3', 'ASTHNOW', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'BLDSUGAR'
]

In [8]:
# Se eliminan columnas irrelevantes
df_cleaned_1 = df_raw.drop(columns=cols_to_drop_1, errors='ignore').copy()

# Se comprueba que las columnas eliminadas y las del actual dataset sumen 330
print(f'Total de columnas: {df_cleaned_1.shape[1] + len(cols_to_drop_1)}')

Total de columnas: 330


### 3.2. Segundo cribado de columnas (34 variables)

In [9]:
#Se comprueban cuales son las columnas que tienen mayor número de nulos
nulls = df_cleaned_1.isnull().sum().sort_values(ascending=False)
nulls_pct = (df_cleaned_1.isnull().sum()/len(df_cleaned_1)*100).sort_values(ascending=False)
pd.DataFrame({"Nulos": nulls, "Porcentaje": nulls_pct})

,Nulos,Porcentaje
ADANXEV,409608,95.399665
ADTHINK,409571,95.391047
ADFAIL,409557,95.387786
ADEAT1,409550,95.386156
ADENERGY,409539,95.383594
ADSLEEP,409534,95.382430
ARTHDIS2,297604,69.313397
BPMEDS,257357,59.939678
AVEDRNK2,223367,52.023244
POORHLTH,209652,48.828955


In [10]:
# Se eliminan columnas con elevado número de NaN y con NaN moderados pero menos relevantes clínicamente
cols_to_drop_2 = ['ADSLEEP', 'ADENERGY', 'ADEAT1', 'ADFAIL', 'ADTHINK', 'ADANXEV', 'ARTHDIS2', 'POORHLTH', 'AVEDRNK2']
df_cleaned_2 = df_cleaned_1.drop(columns=cols_to_drop_2, errors='ignore').copy()

In [11]:
# Se comprueba que las columnas eliminadas y las del actual dataset sumen 34
print(f'Total de columnas: {df_cleaned_2.shape[1] + len(cols_to_drop_2)}')

Total de columnas: 34


### 3.3. Tercer cribado de columnas (25 variables)

In [12]:
# Se crea una clase para visualizar el % de los datos faltantes en las columnas elegidas que no son NaN
class MissingValueAnalyzer:
  def __init__(self, df):
    self.df = df

  # Se calcula el porcentaje, por columna en función de los valores de la lista que representan valores faltantes
  def calculate_percentage(self, columnas, special_values):
    missing_value_results = {'Columna': [], 'Porcentaje': []}

    for col in columnas:
      total = self.df[col].notna().sum()
      count = self.df[col].isin(special_values).sum()
      percentage = (count / total) *100
      missing_value_results['Columna'].append(col)
      missing_value_results['Porcentaje'].append(percentage)

    return pd.DataFrame(missing_value_results)

  # column_groups (diccionario):
  # Clave es el nombre del grupo / Valor es una tupla (columnas, special_values)
  def calculate_all_groups(self, column_groups):
    missing_value_results = {}

    for nombre, (cols, vals) in column_groups.items():
      missing_value_results[nombre] = self.calculate_percentage(cols, vals)

    return missing_value_results

In [13]:
# Se crea el objeto de la clase MissingValueAnalyzer con el df
missing_value_analyzer = MissingValueAnalyzer(df_cleaned_2)

# Se crea el diccionario con nombre del grupo / columnas y valores especiales
column_groups = {
    '7_9': (['GENHLTH', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE',
             'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_PACAT1'], [7, 9]),
    '77_88_99': (['PHYSHLTH', 'MENTHLTH'], [77, 88, 99]),
    '777_999': (['EXEROFT1'], [777, 999]),
    '9': (['_RACE'], [9]),
    '14': (['_AGEG5YR'], [14])
}

# Se crea la variable resultados con el metodo calculate_all_groups del objeto (instancia) missing_value_analyzer
missing_value_results = missing_value_analyzer.calculate_all_groups(column_groups)

# Se muestan los resultados individuales
for nombre, df_resultado in missing_value_results.items():
    print(f"\n=== Resultados para grupo {nombre} ===")
    print(df_resultado)


=== Resultados para grupo 7_9 ===
     Columna  Porcentaje
0    GENHLTH    0.277158
1    BPHIGH4    0.285542
2     BPMEDS    0.171509
3   BLOODCHO    2.138066
4   HAVARTH3    0.587853
5   QLACTLM2    0.715295
6   USEEQUIP    0.248342
7      BLIND    0.377324
8     DECIDE    0.659482
9   DIFFWALK    0.514461
10  DIFFDRES    0.264543
11  DIFFALON    0.431115
12  SMOKE100    0.758683
13  ADDEPEV2    0.457192
14   _PACAT1   12.667691

=== Resultados para grupo 77_88_99 ===
    Columna  Porcentaje
0  PHYSHLTH   64.525257
1  MENTHLTH   70.061720

=== Resultados para grupo 777_999 ===
    Columna  Porcentaje
0  EXEROFT1    0.960703

=== Resultados para grupo 9 ===
  Columna  Porcentaje
0   _RACE      1.6669

=== Resultados para grupo 14 ===
    Columna  Porcentaje
0  _AGEG5YR    1.194336


In [14]:
# Se eliminan columnas con valores "No sabe/Se negó" significativos
cols_to_drop_3 = ['PHYSHLTH', 'MENTHLTH']
df_cleaned_3 = df_cleaned_2.drop(columns=cols_to_drop_3, errors='ignore').copy()

In [15]:
# Se comprueba que las columnas eliminadas y las del actual dataset sumen 25
print(f'Total de columnas: {df_cleaned_3.shape[1] + len(cols_to_drop_3)}')

print(f'Número final de columnas: {df_cleaned_3.shape[1]}')

Total de columnas: 25
Número final de columnas: 23


## 4. Transformación e imputación de variables

### 4.1. Variables que podrían estar relacionadas

In [16]:
# BPMEDS y su relación con BPHIGH
# Se crea una columna que indique si BPMEDS es nulo
df_cleaned_3['BPMEDS_nulos'] = df_cleaned_3['BPMEDS'].isna()

# Se cuenta la distribución de nulos según BPHIGH4
bphigh_distrib = pd.crosstab(df_cleaned_3['BPMEDS_nulos'], df_cleaned_3['BPHIGH4'], margins=True, normalize='columns')
print("Distribución de BPMEDS nulos según BPHIGH4:")
print(bphigh_distrib)

Distribución de BPMEDS nulos según BPHIGH4:
BPHIGH4       1.0  2.0  3.0  4.0  7.0  9.0       All
BPMEDS_nulos                                        
False         1.0  0.0  0.0  0.0  0.0  0.0  0.400604
True          0.0  1.0  1.0  1.0  1.0  1.0  0.599396


In [17]:
# Se asume que los NaNs son debidos a que no tiene tensión alta (por eso no toma esa medicación), así que se sustituye por 2 (categoría de NO)
df_cleaned_3['BPMEDS'] = df_cleaned_3['BPMEDS'].fillna(2)

# Se verifica que no queden NaN en la columna
print(df_cleaned_3['BPMEDS'].isna().sum())

# Se elimina la columna creada de los nulos
df_cleaned_3.drop(columns='BPMEDS_nulos', inplace=True)

0


In [18]:
# EXEROFT1 se sigue el mismo procedimiento que en el caso anterior
df_cleaned_3['EXEROFT1_nulos'] = df_cleaned_3['EXEROFT1'].isna()

# Crosstab para ver relación de nulos en EXEROFT1 con movilidad limitada
for var in ['DIFFALON', 'DIFFDRES', 'DIFFWALK']:
    ct = pd.crosstab(df_cleaned_3['EXEROFT1_nulos'], df_cleaned_3[var], margins=True, normalize='columns')
    print(f"\nDistribución de nulos de EXEROFT1 según {var}:")
    print(ct)

# Se concluye que no tiene significado clínico
# Se elimina la columna creada
df_cleaned_3.drop(columns='EXEROFT1_nulos', inplace=True)


Distribución de nulos de EXEROFT1 según DIFFALON:
DIFFALON             1.0       2.0       7.0       9.0    All
EXEROFT1_nulos                                               
False           0.431847  0.711394  0.475485  0.180828  0.688
True            0.568153  0.288606  0.524515  0.819172  0.312

Distribución de nulos de EXEROFT1 según DIFFDRES:
DIFFDRES             1.0       2.0       7.0       9.0       All
EXEROFT1_nulos                                                  
False           0.398142  0.701415  0.436314  0.095368  0.687038
True            0.601858  0.298585  0.563686  0.904632  0.312962

Distribución de nulos de EXEROFT1 según DIFFWALK:
DIFFWALK             1.0       2.0       7.0       9.0       All
EXEROFT1_nulos                                                  
False           0.465478  0.734292  0.565015  0.177778  0.686392
True            0.534522  0.265708  0.434985  0.822222  0.313608


### 4.2. Tratamiento de variables especiales

In [19]:
#EXEROFT1:

# Para valores, 777 y 999, No sabe/ No contesta, se devuelve el mismo valor. Más adelante se tratará
def decode_exeroft1(x):
    if pd.isna(x):
        return np.nan

    # Si 101–199 → veces por semana (valor - 100)
    elif 101 <= x <= 199:
        return x - 100

    # Si 201–299 → veces por mes (se convierte a veces por semana)
    elif 201 <= x <= 299:
        return (x - 200) / (30.0/7.0)

    # Para valores, 777 y 999, No sabe/ No contesta, mismo valor. Más adelante se tratará
    elif x in (777, 999):
        return x
    # Si hubiese algún valor fuera de rango, se transformará en NaN
    else:
        return np.nan

df_cleaned_3['EXEROFT1'] = df_cleaned_3['EXEROFT1'].apply(decode_exeroft1)

In [20]:
# Valores de EXEROFT1 después del tratamiento
df_cleaned_3['EXEROFT1'].round(2).unique()

array([      nan, 2.800e+00, 1.000e+00, 2.000e+00, 6.000e+00, 3.000e+00,
       4.670e+00, 7.000e+00, 7.770e+02, 1.870e+00, 2.330e+00, 5.000e+00,
       4.000e+00, 3.500e+00, 1.170e+00, 7.000e-01, 2.300e-01, 4.700e-01,
       1.400e+01, 1.400e+00, 9.300e-01, 3.730e+00, 1.630e+00, 6.530e+00,
       5.830e+00, 4.200e+00, 6.770e+00, 1.000e+01, 9.990e+02, 6.300e+00,
       4.900e+00, 5.600e+00, 2.100e+01, 3.270e+00, 3.030e+00, 6.070e+00,
       2.100e+00, 5.370e+00, 8.000e+00, 5.130e+00, 3.970e+00, 2.310e+01,
       1.500e+01, 2.000e+01, 2.400e+01, 5.600e+01, 2.570e+00, 9.330e+00,
       7.230e+00, 1.050e+01, 8.170e+00, 1.200e+01, 2.500e+01, 3.000e+01,
       1.167e+01, 1.800e+01, 3.500e+01, 9.000e+00, 1.100e+01, 5.000e+01,
       9.800e+00, 2.800e+01, 1.700e+01, 4.430e+00, 4.200e+01, 4.000e+01,
       7.470e+00, 1.750e+01, 4.900e+01, 1.600e+01, 7.000e+01, 1.867e+01,
       8.400e+00, 3.100e+01, 1.300e+01, 4.300e+01, 7.500e+01, 8.630e+00,
       3.400e+01, 1.633e+01, 9.900e+01, 1.517e+01, 

In [21]:
#_FRUTSUM y _VEGESUM:

# Cada valor representa raciones/día * 100 → se convierte a raciones reales
df_cleaned_3['_FRUTSUM'] = df_cleaned_3['_FRUTSUM'] / 100
df_cleaned_3['_VEGESUM'] = df_cleaned_3['_VEGESUM'] / 100

In [22]:
# Valores de _FRUTSUM y _VEGESUM después del tratamiento
print(df_cleaned_3['_FRUTSUM'].round(2).unique())
print(df_cleaned_3['_VEGESUM'].round(2).unique())

[5.000e-01 2.400e-01       nan 1.000e+00 2.000e+00 1.290e+00 3.000e+00
 1.140e+00 4.300e-01 8.600e-01 3.400e-01 7.000e-02 1.670e+00 1.430e+00
 2.900e-01 0.000e+00 1.000e-01 1.700e-01 3.600e-01 1.570e+00 4.000e+00
 1.330e+00 6.700e-01 1.070e+00 2.290e+00 3.000e-02 5.300e-01 6.000e-01
 8.300e-01 1.500e+00 2.000e-01 2.600e-01 5.700e-01 4.290e+00 2.140e+00
 3.000e-01 1.830e+00 1.230e+00 5.000e+00 3.570e+00 1.600e-01 7.600e-01
 3.300e-01 6.200e-01 5.130e+00 3.700e-01 1.100e+00 7.200e-01 4.140e+00
 4.400e-01 1.380e+00 1.400e-01 1.300e-01 4.200e-01 7.400e-01 2.430e+00
 8.400e-01 7.800e-01 5.800e-01 1.170e+00 2.000e-02 7.000e+00 7.100e-01
 2.100e-01 1.030e+00 4.000e-01 5.600e-01 2.800e-01 1.130e+00 1.860e+00
 6.000e+00 5.290e+00 1.710e+00 2.070e+00 6.000e-02 8.700e-01 4.700e-01
 9.000e-01 1.280e+00 6.600e-01 3.200e-01 2.300e-01 4.600e-01 2.700e-01
 1.200e+00 3.330e+00 3.900e-01 1.160e+00 8.000e-01 4.860e+00 9.600e-01
 7.300e-01 2.670e+00 3.430e+00 3.100e-01 1.730e+00 7.000e-01 2.570e+00
 2.170

### 4.3. Imputación

In [23]:
# Se eliminan los NaN
df_cleaned_3 = df_cleaned_3.dropna()
df_cleaned_3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 257709 entries, 1 to 429359
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   257709 non-null  float64
 1   BPHIGH4   257709 non-null  float64
 2   BPMEDS    257709 non-null  float64
 3   BLOODCHO  257709 non-null  float64
 4   HAVARTH3  257709 non-null  float64
 5   ADDEPEV2  257709 non-null  float64
 6   DIABETE3  257709 non-null  float64
 7   SEX       257709 non-null  float64
 8   QLACTLM2  257709 non-null  float64
 9   USEEQUIP  257709 non-null  float64
 10  BLIND     257709 non-null  float64
 11  DECIDE    257709 non-null  float64
 12  DIFFWALK  257709 non-null  float64
 13  DIFFDRES  257709 non-null  float64
 14  DIFFALON  257709 non-null  float64
 15  SMOKE100  257709 non-null  float64
 16  EXEROFT1  257709 non-null  float64
 17  _RACE     257709 non-null  float64
 18  _AGEG5YR  257709 non-null  float64
 19  _BMI5CAT  257709 non-null  float64
 20  _FRUTSUM 

In [24]:
# Se definen los valores especiales que indican "No sabe/Se negó"
# Hay 18 (_BMI5CAT, _FRUTSUM, _VEGESUM y SEX no presentan estas categorías)
# DIABETE3 ya se trató para escoger únicamente los valores de Diabetes/No diabetes
valores_invalidos = {
    'GENHLTH': [7, 9],
    'BPHIGH4': [7, 9],
    'BPMEDS': [7, 9],
    'BLOODCHO': [7, 9],
    'HAVARTH3': [7, 9],
    'QLACTLM2': [7, 9],
    'USEEQUIP': [7, 9],
    'BLIND': [7, 9],
    'DECIDE': [7, 9],
    'DIFFWALK': [7, 9],
    'DIFFDRES': [7, 9],
    'DIFFALON': [7, 9],
    'SMOKE100': [7, 9],
    'ADDEPEV2': [7, 9],
    '_PACAT1': [9],
    '_RACE': [9],
    '_AGEG5YR': [14],
    'EXEROFT1': [777, 999]
}

# Se reemplazan estos valores por NaN. Posteriormente se sustituirán por las categorías en función del tipo de variable
for col, vals in valores_invalidos.items():
    df_cleaned_3[col] = df_cleaned_3[col].replace(vals, np.nan)

In [25]:
# Se separa por tipos de variables
categorical_nominal = ['BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_RACE']
categorical_ordinal = ['GENHLTH', '_PACAT1', '_AGEG5YR']
numerical = ['EXEROFT1']

In [26]:
# Se imputa:

# Categóricas nominales. Se imputan con número para mantener el tipo de dato consistente (todos son float)
for col in categorical_nominal:
    df_cleaned_3[col] = df_cleaned_3[col].fillna(-1)

# Categóricas ordinales. Se imputan con número para mantener el tipo de dato consistente (todos son float)
for col in categorical_ordinal:
    df_cleaned_3[col] = df_cleaned_3[col].fillna(-1)

# Numéricas. Se usará la mediana
for col in numerical:
    df_cleaned_3[col] = df_cleaned_3[col].fillna(df_cleaned_3[col].median())

In [27]:
# Se reindexa el dataset
df_cleaned_3 = df_cleaned_3.reset_index(drop=True)

## 5. Exportación

In [28]:
# Se comprueba que está todo correcto antes de exportar
df_cleaned_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257709 entries, 0 to 257708
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   257709 non-null  float64
 1   BPHIGH4   257709 non-null  float64
 2   BPMEDS    257709 non-null  float64
 3   BLOODCHO  257709 non-null  float64
 4   HAVARTH3  257709 non-null  float64
 5   ADDEPEV2  257709 non-null  float64
 6   DIABETE3  257709 non-null  float64
 7   SEX       257709 non-null  float64
 8   QLACTLM2  257709 non-null  float64
 9   USEEQUIP  257709 non-null  float64
 10  BLIND     257709 non-null  float64
 11  DECIDE    257709 non-null  float64
 12  DIFFWALK  257709 non-null  float64
 13  DIFFDRES  257709 non-null  float64
 14  DIFFALON  257709 non-null  float64
 15  SMOKE100  257709 non-null  float64
 16  EXEROFT1  257709 non-null  float64
 17  _RACE     257709 non-null  float64
 18  _AGEG5YR  257709 non-null  float64
 19  _BMI5CAT  257709 non-null  float64
 20  _FRU

In [29]:
# Se exporta dataset limpio
df_cleaned_3.to_csv("/content/drive/MyDrive/TFM/data/cleaned_dataset.csv", index=False)